In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "tensorflow": "tensorflow",
    "keras": "keras",
    "numpy": "numpy",
    "matplotlib": "matplotlib"
}

missing_packages = [
    package_name
    for import_name, package_name in packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing images shape:", x_test.shape)
print("Testing labels shape:", y_test.shape)

In [ ]:
print(x_train[0])
print("Label:", y_train[0])
print("Minimum pixel value:", x_train[0].min())
print("Maximum pixel value:", x_train[0].max())

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.show()

In [ ]:
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")

x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

print("New training shape:", x_train.shape)
print("New testing shape:", x_test.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
y_pred_probabilities = model.predict(x_test)
y_pred = np.argmax(y_pred_probabilities, axis=1)

confusion_matrix = tf.math.confusion_matrix(y_test, y_pred)
confusion_matrix = confusion_matrix.numpy()

plt.figure(figsize=(8, 8))
plt.imshow(confusion_matrix, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()
plt.xticks(np.arange(10))
plt.yticks(np.arange(10))

for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion_matrix[i, j], ha="center", va="center", color="black")

plt.show()

In [ ]:
index = 0

single_image = x_test[index]
actual_label = y_test[index]
single_image_for_prediction = np.expand_dims(single_image, axis=0)

prediction_probabilities = model.predict(single_image_for_prediction)
predicted_label = np.argmax(prediction_probabilities)

plt.imshow(single_image.reshape(28, 28), cmap="gray")
plt.axis("off")
plt.title(f"Actual: {actual_label}, Predicted: {predicted_label}")
plt.show()

print("The model thinks this digit is:", predicted_label)

In [ ]:
model.save("handwritten_digit_recognizer.keras")

print("Model saved successfully.")